In [1]:
import yaml

In [2]:
from anndata import read_h5ad

In [3]:
from os.path import join

In [18]:
import h5py

In [44]:
sample_group_pairs = [
    # AKI vs. HRT
    ('EnrollmentCategory', ('Healthy Reference', 'AKI')),
    # AKI vs. H-CKD. (H-CKD not in enrollment category values anymore. Should I use "Hypertension History" Yes/No column?)
    ('EnrollmentCategory', ('AKI', 'CKD')),
    # D-CKD vs. HRT. (D-CKD not in enrollment category values anymore. Should I use "Diabetes History" Yes/No column?)
    ('EnrollmentCategory', ('CKD', 'Healthy Reference')),
    # Diabetes CKD vs. Hypertension CKD. (DKD nor H-CKD not in enrollment category values anymore. Should I use Yes/No columns?)
    #('EnrollmentCategory', ('DKD', 'H-CKD')),
    # D-CKD vs. HRT
    ('AdjudicatedCategory', ('Diabetic Kidney Disease', 'Healthy Reference')),
    # Acute tubular injury vs. HRT
    ('AdjudicatedCategory', ('Acute Tubular Injury', 'Healthy Reference')),
    # Acute interstitial nephritis vs. HRT
    ('AdjudicatedCategory', ('Acute Interstitial Nephritis', 'Healthy Reference')),
    # Diabetes CKD vs. Hypertension CKD
    ('AdjudicatedCategory', ('Diabetic Kidney Disease', 'Hypertensive Kidney Disease')),
    # ATN vs. AIN
    ('AdjudicatedCategory', ('Acute Interstitial Nephritis', 'Acute Tubular Injury')),

    # TODO: use Diabetes History and Hypertension History columns here.
]
cell_type_cols = [
    "subclass_l1",
    "subclass_l2",
]

In [45]:
adata_path = join("data", "raw", "kpmp-sc-aug-2026", "KPMP_PREMIERE_SC_version2_ForExplorer_RemovedBatchEffect_Final2025.h5ad")
clinical_path = join("data", "raw", "kpmp-sc-aug-2026", "20260618_OpenAccessClinicalData.csv")

In [46]:
adata = read_h5ad(adata_path)

In [47]:
adata

AnnData object with n_obs × n_vars = 348984 × 33298
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'KPMPID', 'barcode', 'SpecimenID', 'LibraryID', 'InternalID', 'Tissue.Type', 'Protocol', 'Age', 'Gender', 'Race', 'assay', 'tissue', 'organism', 'disease', 'SampleID', 'umap_1', 'umap_2', 'cell_type', 'development_stage', 'development_stage_ontology_term_id', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'cell_type_ontology_term_id', 'DataSourceID', 'state.l2', 'state.l1', 'class', 'SubclassLevel2_FullName', 'subclass.level1', 'subclass.level2', 'cluster', 'EnrollmentCategory', 'AdjudicationCategory', 'self_reported_race', 'diabetes_history', 'hypertension_history', 'condition'
    var: 'vf_vst_counts_mean', 'vf_vst_counts_variance', 'vf_vst_counts_variance.expected', 'vf_vst_counts_variance.standardized', 'vf_vst_counts_variable', 'vf_vst_counts_rank', 'var.features', 'var.features.rank'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'

In [48]:
# Fix categorical columns
f = h5py.File(adata_path)

def clean_category_chars(s):
    # The cell types have odd characters
    return s.replace("ï", "i").replace("⁺", "_pos").replace("ʰⁱ", "_hi")
    
def fix_categorical_column(colname):
    if adata.obs[colname].dtype.kind == "i":
        # This is an integer column
        try:
            categories = f[f"/obs/__categories/{colname}"][()].astype(str)
        except:
            # UnicodeDecodeError
            categories = [ b.decode("utf-8") for b in f[f"/obs/__categories/{colname}"][()] ]

        categories = [ clean_category_chars(c) for c in categories ]
        return adata.obs[colname].apply(lambda i: categories[i])

adata.obs["subclass.level1"] = fix_categorical_column("subclass.level1")
adata.obs["subclass.level2"] = fix_categorical_column("subclass.level2")

adata.obs = adata.obs.rename(columns={"subclass.level1": "subclass_l1", "subclass.level2": "subclass_l2"})

In [49]:
clean_adata_path = join("data", "raw", "kpmp-sc-aug-2026", "KPMP_PREMIERE_SC_version2_ForExplorer_RemovedBatchEffect_Final2025.clean.h5ad")
adata.write_h5ad(clean_adata_path)

In [50]:
adata.obs["subclass_l1"].unique().tolist()

['IC',
 'EC',
 'CNT',
 'PT',
 'DCT',
 'PC',
 'Myeloid',
 'PEC',
 'TAL',
 'VSMC/P',
 'Lymphoid',
 'POD',
 'DTL',
 'FIB',
 'SC/NEU',
 'ATL',
 'PL']

In [9]:
adata.obs["SubclassLevel2_FullName"].unique().tolist()

['Collecting Duct Intercalated Cell Type A',
 'Postcapillary Venule Endothelial Cell',
 'Connecting Tubule Cell',
 'Proximal Tubule Epithelial Cell Segment 2/Segment 3',
 'Distal Convoluted Tubule Cell Type ',
 'Glomerular Capillary Endothelial Cell',
 'Adaptive / Maladaptive / Repairing Glomerular Capillary Endothelial Cell',
 'Proximal Tubule Epithelial Cell Segment 1/Segment 2',
 'Adaptive / Maladaptive / Repairing Proximal Tubule Epithelial Cell',
 'Transitional Principal-Intercalated Cell',
 'Collecting Duct Intercalated Cell Type B',
 'Monocyte-Derived C3+ Macrophage',
 'Adaptive/ Maladaptive  Connecting Tubule Principal cell',
 'Adaptive / Maladaptive / Repairing Collecting Duct Intercalated Cell Type A',
 'Connecting Tubule Principal cell',
 ' Parietal Epithelial Cell',
 'Degenerative Proximal Tubule Epithelial Cell Segment ',
 'Cortical Thick Ascending Limb Cell',
 'Degenerative Glomerular Capillary Endothelial Cell',
 'Adaptive / Maladaptive / Repairing Peritubular Capillary 

In [51]:
cell_types = {}
for colname in cell_type_cols:
    cell_types[colname] = sorted(adata.obs[colname].unique().tolist(), key=lambda v: v.lower())

In [52]:
sample_group_pairs_dict = [
    { "colname": t[0], "lhs": t[1][0], "rhs": t[1][1] }
    for t in sample_group_pairs
]

In [70]:
import pandas as pd
# Join adata.obs with clinical data from CSV
clinical_data = pd.read_csv(clinical_path)

adata.obs = adata.obs.merge(clinical_data, left_on="SpecimenID", right_on="Participant ID", how="left")

# We could have done a left join, but then we would have to filter out samples that do not have clinical data later.
# We also do not want strings to be converted to NaN, as these cause Zarr writing errors like "TypeError: expected unicode string, found nan".

# This effectively does an inner join. We cannot use how="inner", since this would only affect adata.obs, and not other anndata fields.
has_clinical_data = ~adata.obs["Participant ID"].isna()
adata = adata[has_clinical_data, :].copy()

/Users/mkeller/.local/share/uv/python/cpython-3.12.5-macos-aarch64-none/lib/python3.12/functools.py:907: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [71]:
patient_ids = adata.obs["Participant ID"].unique().tolist()
specimen_ids = adata.obs["SampleID"].unique().tolist()

In [73]:
adata.shape

(25531, 33298)

In [72]:
yaml_output = yaml.dump({
    "cell_types": cell_types,
    "sample_group_pairs": sample_group_pairs_dict,
    "patient_ids": patient_ids,
    "specimen_ids": specimen_ids,
}, default_flow_style=False)
print(yaml_output)

cell_types:
  subclass_l1:
  - ATL
  - CNT
  - DCT
  - DTL
  - EC
  - FIB
  - IC
  - Lymphoid
  - Myeloid
  - PC
  - PEC
  - PL
  - POD
  - PT
  - SC/NEU
  - TAL
  - VSMC/P
  subclass_l2:
  - aATL
  - aCCD-PC
  - aCNT
  - aCNT-PC
  - aDCT
  - aDTL
  - aEC-DVR
  - aEC-GC
  - aEC-PTC
  - aIC-A
  - angEC-PTC
  - aOMCD-PC
  - aPT
  - aTAL
  - ATL
  - B activated
  - B memory
  - B naive
  - C-TAL
  - C/M-TAL
  - CCD-PC
  - CD16+ NK
  - CD16- NK
  - CD8+ T-CYT
  - CD8+ T-STR
  - CD8+ TEM
  - CD8+ TEMRA
  - CD8+ TRM
  - cDC1
  - cDC2
  - cMON
  - CNT
  - CNT-PC
  - cycMAC
  - cycPT
  - cycT
  - dATL
  - dCCD-PC
  - DCT
  - dDN
  - dEC-EA
  - dEC-GC
  - dEC-PTC
  - dM-TAL
  - dPOD
  - dPT
  - dTAL
  - DTL1
  - DTL2
  - dVSMC
  - EC-AA
  - EC-AVR
  - EC-DVR
  - EC-EA
  - EC-GC
  - EC-LYM
  - EC-PCV
  - EC-PTC
  - EC-V
  - frPT
  - frTAL
  - IC-A
  - IC-B
  - ILC3
  - M-TAL
  - MAIT
  - MAST
  - MC
  - MD
  - mDC
  - moFAM
  - moMAC-C3_pos
  - moMAC-CXCL10_pos
  - moMAC-HBEGF_pos
  - moMAC-LUCA

In [10]:
import numpy as np
should_subset = True
if should_subset:
    print("SUBSETTING")
    # subset using random sample so that multiple sample groups are represented to enable comparison
    np.random.seed(1)
    obs_subset = np.random.choice(adata.obs.index.tolist(), size=20_000, replace=False).tolist()
    var_slice = slice(0, 5_000)
    adata = adata[obs_subset, var_slice].copy()

SUBSETTING


In [11]:
# CLEANUP FROM SCRIPT
import pandas as pd
# Join adata.obs with clinical data from CSV
clinical_data = pd.read_csv(clinical_path)

adata.obs = adata.obs.merge(clinical_data, left_on="patient", right_on="Participant ID", how="left")

# We could have done a left join, but then we would have to filter out samples that do not have clinical data later.
# We also do not want strings to be converted to NaN, as these cause Zarr writing errors like "TypeError: expected unicode string, found nan".

# This effectively does an inner join. We cannot use how="inner", since this would only affect adata.obs, and not other anndata fields.
has_clinical_data = ~adata.obs["Participant ID"].isna()
adata = adata[has_clinical_data, :].copy()

#print(adata.obs.head())

adata.obs["Primary Adjudicated Category"] = adata.obs["Primary Adjudicated Category"].fillna("NA")

# Cleanup of sample-level data
def clean_adjudicated_category(row):
    if row["Primary Adjudicated Category"] != "NA":
        return row["Primary Adjudicated Category"]
    else:
        # The row was empty, so perhaps this sample has not yet been adjudicated.
        # However, we also need to check that this was not a "Healthy Reference" sample,
        # as these never go through the adjudication process.
        if row["Enrollment Category"] in ["Healthy Reference"]:
            return "Healthy Reference"
        return ""
adata.obs["AdjudicatedCategory"] = adata.obs.apply(clean_adjudicated_category, axis='columns')
adata.obs["EnrollmentCategory"] = adata.obs["Enrollment Category"]

# TODO: process other clinical columns? Sex, age group, etc.

adata.obs = adata.obs.rename(columns={"subclass.l1": "subclass_l1", "subclass.l2": "subclass_l2", "subclass.l3": "subclass_l3"})

for colname in adata.obs.columns:
    if pd.api.types.is_string_dtype(adata.obs[colname]) or str(adata.obs[colname].dtype) == "object":
        print(f"Filling NAs in string column {colname} with 'NA'")
        adata.obs[colname] = adata.obs[colname].fillna("NA")
    else:
        print(f"Not filling NAs in non-string column {colname} of type {adata.obs[colname].dtype}")

# Column names cannot contain slashes
adata.obs = adata.obs.rename(columns=dict(zip(adata.obs.columns, [c.replace("/", " per ") for c in adata.obs.columns])))


Filling NAs in string column library_id with 'NA'
Not filling NAs in non-string column nCount_RNA of type float64
Not filling NAs in non-string column nFeature_RNA of type float64
Not filling NAs in non-string column percent.er of type float64
Not filling NAs in non-string column percent.mt of type float64
Filling NAs in string column experiment_id with 'NA'
Filling NAs in string column specimen with 'NA'
Filling NAs in string column patient with 'NA'
Filling NAs in string column region with 'NA'
Not filling NAs in non-string column percent.cortex of type Int32
Not filling NAs in non-string column percent.medulla of type Int32
Filling NAs in string column subclass_l3 with 'NA'
Filling NAs in string column subclass_l2 with 'NA'
Filling NAs in string column subclass_l1 with 'NA'
Filling NAs in string column class with 'NA'
Not filling NAs in non-string column UMAP_1 of type float64
Not filling NAs in non-string column UMAP_2 of type float64
Filling NAs in string column Participant ID wit

/Users/mkeller/.local/share/uv/python/cpython-3.11.9-macos-aarch64-none/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [12]:
adata.layers['counts']

<18694x5000 sparse matrix of type '<class 'numpy.float64'>'
	with 1101495 stored elements in Compressed Sparse Column format>

In [13]:
import decoupler as dc

/Users/mkeller/research/dbmi/vitessce/compasce/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
# References:
# - https://pertpy.readthedocs.io/en/stable/tutorials/notebooks/differential_gene_expression.html#pseudobulks
# - https://decoupler.readthedocs.io/en/latest/api/generated/decoupler.pp.pseudobulk.html

In [15]:
pdata = dc.pp.pseudobulk(adata, sample_col="specimen", groups_col="subclass_l1", layer="counts", mode="sum", empty=True, verbose=False)

In [16]:
del adata

In [17]:
pdata

AnnData object with n_obs × n_vars = 2916 × 4496
    obs: 'specimen', 'subclass_l1', 'patient', 'region', 'percent.cortex', 'percent.medulla', 'class', 'Participant ID', 'Tissue Source', 'Protocol', 'Sample Type', 'Enrollment Category', 'Primary Adjudicated Category', 'Sex', 'Age (Years) (Binned)', 'Race', 'KDIGO Stage', 'Baseline eGFR (ml per min per 1.73m2)', 'Baseline eGFR (ml per min per 1.73m2) (Binned)', 'Proteinuria (mg) (Binned)', 'A1c (%) (Binned)', 'Albuminuria (mg) (Binned)', 'Diabetes History', 'Diabetes Duration (Years)', 'Hypertension History', 'Hypertension Duration (Years)', 'On RAAS Blockade', 'AdjudicatedCategory', 'EnrollmentCategory', 'psbulk_cells', 'psbulk_counts'
    var: 'gene_symbol', 'ensembl_id'
    layers: 'psbulk_props'

In [18]:
import pertpy as pt

/Users/mkeller/research/dbmi/vitessce/compasce/.venv/lib/python3.11/site-packages/anndata/__init__.py:44: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/Users/mkeller/research/dbmi/vitessce/compasce/.venv/lib/python3.11/site-packages/anndata/__init__.py:44: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/Users/mkeller/research/dbmi/vitessce/compasce/.venv/lib/python3.11/site-packages/anndata/__init__.py:44: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/Users/mkeller/research/dbmi/vitessce/compasce/.venv/lib/python3.11/site-packages/anndata/experimental/__init__.py:48: FutureWarning: Importing CSC

In [19]:
pdata.obs["subclass_l1"].unique()

['ATL', 'Ad', 'CNT', 'DCT', 'DTL', ..., 'POD', 'PT', 'PapE', 'TAL', 'VSM/P']
Length: 18
Categories (18, object): ['ATL', 'Ad', 'CNT', 'DCT', ..., 'PT', 'PapE', 'TAL', 'VSM/P']

In [20]:
pdata.obs["specimen"].unique()

['18-142-3-M2', '18-162-2-M2', '18-312-2-M2', '446_B1', '446_B3', ..., 'S-2305-006657_D1_N1', 'S-2306-001149_D1_N1', 'S-2306-012843_D1_N1', 'S-2307-011657_D1_N1', 'S-2307-011845_D1_N1']
Length: 162
Categories (162, object): ['18-142-3-M2', '18-162-2-M2', '18-312-2-M2', '446_B1', ..., 'S-2306-001149_D1_N1', 'S-2306-012843_D1_N1', 'S-2307-011657_D1_N1', 'S-2307-011845_D1_N1']

In [21]:
18*160

2880

In [22]:
(pdata.obs['psbulk_cells'] != 0.0).sum()

1764

In [23]:
(pdata.obs['psbulk_counts'] != 0.0).sum()

1764

In [24]:
(pdata.obs['psbulk_cells'] == 0.0).sum()

1152

In [25]:
(pdata.obs['psbulk_counts'] == 0.0).sum()

1152

In [26]:
pdata[0,:]

View of AnnData object with n_obs × n_vars = 1 × 4496
    obs: 'specimen', 'subclass_l1', 'patient', 'region', 'percent.cortex', 'percent.medulla', 'class', 'Participant ID', 'Tissue Source', 'Protocol', 'Sample Type', 'Enrollment Category', 'Primary Adjudicated Category', 'Sex', 'Age (Years) (Binned)', 'Race', 'KDIGO Stage', 'Baseline eGFR (ml per min per 1.73m2)', 'Baseline eGFR (ml per min per 1.73m2) (Binned)', 'Proteinuria (mg) (Binned)', 'A1c (%) (Binned)', 'Albuminuria (mg) (Binned)', 'Diabetes History', 'Diabetes Duration (Years)', 'Hypertension History', 'Hypertension Duration (Years)', 'On RAAS Blockade', 'AdjudicatedCategory', 'EnrollmentCategory', 'psbulk_cells', 'psbulk_counts'
    var: 'gene_symbol', 'ensembl_id'
    layers: 'psbulk_props'

In [27]:
pdata.X.shape

(2916, 4496)

In [31]:
conditions_with_nonzero_expression = (pdata.obs['psbulk_counts'] != 0.0)

In [32]:
genes_to_keep = pdata.X.sum(axis=0) >= 10

In [33]:
nonzero_pdata = pdata[conditions_with_nonzero_expression, genes_to_keep].copy()

In [34]:
nonzero_pdata.shape

(1764, 3200)

In [37]:
pds2 = pt.tl.PyDESeq2(adata=nonzero_pdata, design=f"~subclass_l1")
pds2.fit(n_cpus=4)

Fitting size factors...
/Users/mkeller/research/dbmi/vitessce/compasce/.venv/lib/python3.11/site-packages/pydeseq2/dds.py:532: UserWarning: Every gene contains at least one zero, cannot compute log geometric means. Switching to iterative mode.
  self.fit_size_factors(


Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 0.48 seconds.

Fitting MAP dispersions...
... done in 0.96 seconds.



KeyboardInterrupt: 

In [ ]:
df = pds2.test_contrasts(pds2.contrast(column="subclass_l1", baseline="POD", group_to_compare=""))